# Steering layer sweep

Top-3 AUROC layers (8, 14, 16) from the sklearn probe evaluation x factors {1.2, 2.0}, paired GSM8K eval. Each layer ships TRAIN-derived centroid-difference vectors (`steering_layer{L}_vector_train.npy`, Issue #5) and l2-normalized sklearn LR probe weights (`steering_layer{L}_probe_weights.npy`). Cells 4-6 fetch them from the local checkout, fall back to the GitHub raw mirror, and only rebuild from `train_all_layers_acts.pth` if both are unreachable.


## 1. Install

In [ ]:
# Run once per runtime. Safe to skip if already installed.
%pip install -q "transformers>=4.44" datasets accelerate matplotlib seaborn scikit-learn tqdm pandas


## 2. Imports + seed

In [ ]:
import json
import os
import random
import re
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.bfloat16
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"
    DTYPE = torch.float32
else:
    DEVICE = "cpu"
    DTYPE = torch.float32

RUN_TAG = "layer_sweep"
RESULTS_DIR = Path("nb_results") / RUN_TAG
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"device={DEVICE} dtype={DTYPE} run_tag={RUN_TAG}")
print(f"results -> {RESULTS_DIR.resolve()}")


## 3. Config

In [ ]:
LAYERS = [8, 14, 16]
FACTORS = [1.2, 2.0]
MAX_EXAMPLES = 100
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.6
TOP_P = 0.9
FORCE_RERUN = False


## 4. Probe loading scaffold

In [ ]:
# Probe location.  Fallback order: local repo -> GitHub raw -> manual upload.
LOCAL_PROBE_DIR = Path("artifacts/cached3/sklearn/steering_configs")
PROBE_RAW_URL_BASE = (
    "https://raw.githubusercontent.com/"
    "stvngo/Pivotal-Token-Representation-Learning/"
    "main/artifacts/cached3/sklearn/steering_configs/"
)


In [ ]:
import urllib.request


def _fetch(local_path: Path, dest_dir: Path) -> Path:
    """Return a usable path for ``local_path``. If the local copy is missing,
    download the file from ``PROBE_RAW_URL_BASE`` into ``dest_dir`` and return
    that. Raises ``RuntimeError`` only if both fallbacks fail."""
    if local_path.exists():
        return local_path
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / local_path.name
    if dest.exists():
        return dest
    try:
        urllib.request.urlretrieve(PROBE_RAW_URL_BASE + local_path.name, dest)
        return dest
    except Exception as exc:
        raise RuntimeError(
            f"Could not locate {local_path.name}. Tried local: {local_path} "
            f"and remote: {PROBE_RAW_URL_BASE + local_path.name}. ({exc})"
        ) from exc


def load_probe(
    layer: int,
    local_dir: Path = LOCAL_PROBE_DIR,
    *,
    prefer_train_caa: bool = True,
):
    """Load the steering bundle for ``layer``.

    Closes Issue #5 (CAA on val → CAA on train): when ``prefer_train_caa`` is
    true and ``steering_layer{L}_vector_train.npy`` is available locally or on
    the GitHub mirror, the centroid-difference vector returned is the one
    derived from the *training* slice the LR probe was actually fit on. The
    legacy ``steering_layer{L}_vector.npy`` (val-derived) is used only as a
    last-resort fallback.

    Returns a dict containing both the (raw, l2-normalized) centroid-diff
    vector and — when shipped — the (raw, l2-normalized) sklearn LR probe
    weight vector. Cosine similarities between the two directions are also
    reported, since the ``arena_linear_probes`` analysis uses that as a sanity
    check.
    """
    cfg_name   = f"steering_layer{layer}.json"
    train_name = f"steering_layer{layer}_vector_train.npy"
    val_name   = f"steering_layer{layer}_vector.npy"
    pw_name    = f"steering_layer{layer}_probe_weights.npy"
    pb_name    = f"steering_layer{layer}_probe_biases.npy"
    dest = RESULTS_DIR / "probes"

    cfg_path = _fetch(local_dir / cfg_name, dest)
    cfg = json.loads(cfg_path.read_text())

    vec_path, vec_source = None, None
    if prefer_train_caa:
        try:
            vec_path = _fetch(local_dir / train_name, dest)
            vec_source = "centroid_diff_train"
        except RuntimeError:
            vec_path = None
    if vec_path is None:
        vec_path = _fetch(local_dir / val_name, dest)
        vec_source = "centroid_diff_val"
    vec = np.load(vec_path).astype(np.float32).reshape(-1)
    vec_norm = float(np.linalg.norm(vec))
    vec_unit = vec / max(1e-12, vec_norm)

    pw, pw_unit, pw_norm, pw_source = None, None, None, None
    try:
        pw_path = _fetch(local_dir / pw_name, dest)
        pw = np.load(pw_path).astype(np.float32).reshape(-1)
        pw_norm = float(np.linalg.norm(pw))
        pw_unit = pw / max(1e-12, pw_norm)
        pw_source = "sklearn_logreg_C1.0"
    except RuntimeError:
        pass

    pb = None
    try:
        pb_path = _fetch(local_dir / pb_name, dest)
        pb = np.load(pb_path).astype(np.float32).reshape(-1)
    except RuntimeError:
        pass

    cosine_pw_caa = (
        float(np.dot(pw, vec) / (pw_norm * vec_norm + 1e-12))
        if pw is not None else None
    )
    cfg.update({
        "vector_path_resolved": str(vec_path),
        "vector_source": vec_source,
        "vector_norm": vec_norm,
        "probe_weight_source": pw_source,
        "probe_weight_norm": pw_norm,
        "cosine_probe_to_caa": cosine_pw_caa,
    })
    return {
        "cfg": cfg,
        "vector": vec,
        "vector_unit": vec_unit,
        "probe_weights": pw,
        "probe_weights_unit": pw_unit,
        "probe_biases": pb,
    }


In [ ]:
# --- Google Colab: upload your own probe --------------------------------
# Uncomment if the automatic fallback above fails.
#
# from google.colab import files
# uploaded = files.upload()
# print('Uploaded:', list(uploaded.keys()))


## 5. Prepare layer-8 vector (defensive fallback)

Layers 8, 14 and 16 each ship `steering_layer{L}_vector_train.npy` (TRAIN CAA, Issue #5) plus `steering_layer{L}_probe_weights.npy` (sklearn LR probe weights, normalized at use site) under `artifacts/cached3/sklearn/steering_configs/`. The `load_probe` cell above already handles fetching these from the GitHub mirror when running on Colab; the cell below is a defensive fallback for hosts where neither the local checkout nor the raw URL is reachable. It rebuilds the L8 train CAA from `train_all_layers_acts.pth` if available.

In [ ]:
def maybe_build_layer8_vector():
    """Defensive fallback. The shipped artifacts (`steering_layer8.json`,
    `_vector_train.npy`, `_vector.npy`, `_probe_weights.npy`,
    `_probe_biases.npy`) are normally fetched by `load_probe` directly. This
    cell only runs the rebuild path if both the local checkout and the GitHub
    mirror are unreachable."""
    cfg_path = LOCAL_PROBE_DIR / "steering_layer8.json"
    train_vec = LOCAL_PROBE_DIR / "steering_layer8_vector_train.npy"
    if cfg_path.exists() and train_vec.exists():
        print("layer-8 train CAA + config already present in repo.")
        return None
    train_cache = Path("data/cached_activations_3 (without query positions)/train_all_layers_acts.pth")
    if train_cache.exists():
        try:
            from probe_pipeline.activations import centroid_diff_from_cache  # noqa
        except Exception:
            print("probe_pipeline import failed; cannot rebuild from train cache.")
            return None
        v = centroid_diff_from_cache(train_cache, layer=8)
        out_dir = RESULTS_DIR / "probes"; out_dir.mkdir(parents=True, exist_ok=True)
        np.save(out_dir / "steering_layer8_vector_train.npy", v.astype(np.float32))
        cfg = {"backend": "sklearn", "layer": 8, "vector_type": "centroid_diff",
               "hidden_dim": int(v.shape[0]),
               "vector_norm": float(np.linalg.norm(v)),
               "vector_path": "steering_layer8_vector_train.npy",
               "split": "train"}
        (out_dir / "steering_layer8.json").write_text(json.dumps(cfg, indent=2))
        print(f"Rebuilt layer-8 TRAIN CAA from cache: norm={cfg['vector_norm']:.4f}")
        return out_dir
    print("layer-8 train cache not found locally; load_probe will fetch from GitHub.")
    return None

_layer8_local = maybe_build_layer8_vector()


## 6. Load all three probes

In [ ]:
probes = {}
for L in LAYERS:
    fallback_dir = RESULTS_DIR / "probes"
    bundle = (
        load_probe(L, local_dir=fallback_dir)
        if (fallback_dir / f"steering_layer{L}.json").exists()
        else load_probe(L)
    )
    cfg = bundle["cfg"]
    vec = bundle["vector"]                     # raw centroid-diff (TRAIN CAA preferred)
    vec_unit = bundle["vector_unit"]           # l2-normalized direction
    pw = bundle["probe_weights"]               # raw sklearn LR weights or None
    pw_unit = bundle["probe_weights_unit"]     # l2-normalized direction or None
    probes[L] = {
        "cfg": cfg,
        "vector": vec,
        "vector_unit": vec_unit,
        "tensor": torch.from_numpy(vec).to(torch.float32),
        "tensor_unit": torch.from_numpy(vec_unit).to(torch.float32),
        "probe_weights": pw,
        "probe_weights_unit": pw_unit,
        "probe_tensor": torch.from_numpy(pw).to(torch.float32) if pw is not None else None,
        "probe_tensor_unit": torch.from_numpy(pw_unit).to(torch.float32) if pw_unit is not None else None,
    }
    pw_summary = (
        f" pw_norm={cfg['probe_weight_norm']:.3f} cos(pw,caa)={cfg['cosine_probe_to_caa']:+.3f}"
        if pw is not None else " probe_weights=N/A"
    )
    print(
        f"layer {L}: caa_source={cfg['vector_source']} caa_norm={cfg['vector_norm']:.4f}"
        f" hidden_dim={cfg['hidden_dim']}{pw_summary}"
    )


## 7. Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-0.6B"

_t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
).to(DEVICE)
model.eval()
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
print(
    f"Loaded {MODEL_NAME} on {DEVICE} ({DTYPE}) "
    f"in {time.time() - _t0:.1f}s | hidden_size={model.config.hidden_size} "
    f"layers={model.config.num_hidden_layers}"
)


In [ ]:
for L, p in probes.items():
    assert p["vector"].shape[0] == model.config.hidden_size, f"layer {L} hidden_dim mismatch"


## 8. Hooks + parser + dataset

In [ ]:
import torch
import torch.nn as nn

# Pull the canonical, off-by-one-fixed hook factory from probe_pipeline.steering
# (issues #2 + #7 in docs/issues.md). If the local checkout isn't on sys.path
# (e.g. fresh Colab runtime), fall back to a vendored copy with the same fix.
try:
    from probe_pipeline.steering import _get_decoder_layer, make_hook  # noqa: F401
    print("[hooks] using probe_pipeline.steering (post-Issue-#2/#7 fix)")
except Exception:
    print("[hooks] probe_pipeline not importable, vendoring local fix")

    def _get_decoder_layer(mdl: nn.Module, layer_idx: int) -> nn.Module:
        """Returns the module whose forward-hook output IS hidden_states[layer_idx].

        Post-Issue-#2: layers[layer_idx-1] for layer_idx>=1; embed_tokens for 0.
        """
        if hasattr(mdl, "model") and hasattr(mdl.model, "layers"):
            if layer_idx == 0:
                return mdl.model.embed_tokens
            return mdl.model.layers[layer_idx - 1]
        if hasattr(mdl, "transformer") and hasattr(mdl.transformer, "h"):
            if layer_idx == 0:
                return mdl.transformer.wte
            return mdl.transformer.h[layer_idx - 1]
        raise AttributeError(f"Cannot find decoder layers in {type(mdl)}")

    def make_hook(vector, coef, mode="additive_normalized", position_mask=None):
        if mode not in {"additive_raw", "additive_normalized", "projection"}:
            raise ValueError(f"unknown mode {mode!r}")

        def hook(_module, _args, output):
            hidden = output[0] if isinstance(output, tuple) else output
            v = vector.to(hidden.device).to(hidden.dtype)
            if v.dim() == 1:
                v_full = v.view(1, 1, -1)
            else:
                v_full = v
            if mode == "additive_raw":
                delta = coef * v_full
            elif mode == "additive_normalized":
                v_hat = v / (v.norm() + 1e-8)
                v_hat_b = v_hat.view(1, 1, -1)
                h_norm = hidden.norm(dim=-1, keepdim=True)
                delta = coef * h_norm * v_hat_b
            else:  # projection
                v_hat = v / (v.norm() + 1e-8)
                v_hat_b = v_hat.view(1, 1, -1)
                proj = (hidden * v_hat_b).sum(dim=-1, keepdim=True)
                delta = (coef - 1.0) * proj * v_hat_b
            if position_mask is not None:
                if position_mask.dim() == 1:
                    pm = position_mask.view(1, -1, 1)
                else:
                    pm = position_mask.unsqueeze(-1)
                delta = delta * pm.to(device=hidden.device, dtype=hidden.dtype)
            new_hidden = hidden + delta
            if isinstance(output, tuple):
                return (new_hidden,) + output[1:]
            return new_hidden
        return hook


# ---- thin wrappers that preserve legacy notebook entry points ---------------

# STEERING_MODE controls which convention the *_steering helpers below use.
# Default "additive_normalized" follows the Issue-#7 plan ("canonical default
# for new runs"). Set to "additive_raw" in the notebook BEFORE re-running its
# steered cells if you need the legacy convention for an apples-to-apples
# overlay with the pre-fix runs in artifacts/notebook_runs/REGISTRY.md.
STEERING_MODE = "additive_normalized"


def register_steering(mdl, layer: int, vector, coef: float, mode: str | None = None):
    """Register a single forward hook at ``layer``. ``coef`` is the value we
    pass into ``make_hook``: for ``additive_raw`` that's the legacy
    ``factor - 1.0``; for ``additive_normalized`` it's the unitless per-token
    coefficient on ``||h|| * v_hat``; for ``projection`` it's ``alpha`` (where
    alpha=1 is identity and alpha=0 ablates).
    """
    return [
        _get_decoder_layer(mdl, layer).register_forward_hook(
            make_hook(vector, coef, mode=(mode or STEERING_MODE))
        )
    ]


def register_additive_hooks(mdl, injections, mode: str | None = None):
    """One additive hook per (layer_idx, vector, coef) tuple."""
    handles = []
    for layer_idx, vector, coef in injections:
        handles.append(
            _get_decoder_layer(mdl, layer_idx).register_forward_hook(
                make_hook(vector, coef, mode=(mode or STEERING_MODE))
            )
        )
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


# Probe-weights notebook keeps its own additive/projection helper names so
# section 11/12/15 cells in that notebook don't need rewriting.
def make_additive_hook(vector, alpha):
    return make_hook(vector, alpha, mode="additive_normalized")


def make_projection_hook(unit_vector, alpha):
    return make_hook(unit_vector, alpha, mode="projection")


def register_additive(mdl, layer: int, vector, alpha: float):
    return [_get_decoder_layer(mdl, layer).register_forward_hook(
        make_additive_hook(vector, alpha)
    )]


def register_projection(mdl, layer: int, unit_vector, alpha: float):
    return [_get_decoder_layer(mdl, layer).register_forward_hook(
        make_projection_hook(unit_vector, alpha)
    )]


In [ ]:
# Convention check (Issue #2 in docs/issues.md): the forward-hook output of
# `_get_decoder_layer(model, L)` must equal `outputs.hidden_states[L]`. If
# this asserts, the off-by-one fix is in effect and every steering hook below
# modifies the same residual tensor the layer-L probe was trained on.
import torch as _torch  # local alias avoids polluting the notebook namespace

def _convention_check(_model, _layer: int) -> bool:
    captured = {}

    def _cap(_m, _i, _out):
        h = _out[0] if isinstance(_out, tuple) else _out
        captured["pre"] = h.detach()
        return _out

    target = _get_decoder_layer(_model, _layer)
    h = target.register_forward_hook(_cap)
    try:
        _device = next(_model.parameters()).device
        ids = _torch.tensor([[1, 2, 3]], device=_device)
        with _torch.no_grad():
            outs = _model(input_ids=ids, output_hidden_states=True)
    finally:
        h.remove()
    expected = outs.hidden_states[_layer]
    if captured.get("pre") is None:
        return False
    if captured["pre"].shape != expected.shape:
        return False
    return bool(_torch.allclose(captured["pre"], expected, atol=1e-4, rtol=1e-3))

# Pick a layer to check. Notebooks define one of these names.
_probe_layer_for_check = next(
    (v for v in (globals().get("LAYER"), globals().get("PROBE_LAYER"),
                 globals().get("BEST_LAYER"), globals().get("CODELION_LAYER"))
     if isinstance(v, int)),
    14,
)
CONVENTION_CHECK_PASSED = _convention_check(model, _probe_layer_for_check)
print(f"[convention check] hook output == hidden_states[{_probe_layer_for_check}]: "
      f"{CONVENTION_CHECK_PASSED}")
assert CONVENTION_CHECK_PASSED, (
    "Off-by-one regression: hook on layers[L-1] no longer equals "
    "hidden_states[L]. See docs/issues.md Issue #2."
)


In [ ]:
def extract_gsm8k_answer(text: str):
    m = re.search(r"####\s*([+-]?\d+(?:,\d{3})*(?:\.\d+)?)", text)
    if m:
        return m.group(1).replace(",", "").strip()
    nums = re.findall(r"[-+]?\d*\.?\d+", text)
    return nums[-1] if nums else None


def is_correct(pred, gt):
    if pred is None or gt is None:
        return False
    p = pred.strip().replace(",", "")
    g = gt.strip().replace(",", "")
    try:
        return float(p) == float(g)
    except ValueError:
        return p == g


def compute_metrics(results):
    n = len(results)
    if n == 0:
        return {"accuracy": 0.0, "f1": 0.0, "parse_rate": 0.0, "correct": 0, "n": 0}
    correct = sum(1 for r in results if r["correct"])
    parsed = sum(1 for r in results if r["predicted"] is not None)
    tp = correct
    fp = parsed - correct
    fn = n - correct
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {
        "accuracy": correct / n,
        "f1": f1,
        "parse_rate": parsed / n,
        "correct": correct,
        "n": n,
    }


# Inline parser sanity checks (mirror tests/test_answer_extraction.py).
assert extract_gsm8k_answer("#### 42") == "42"
assert extract_gsm8k_answer("He has #### 1,234 apples.") == "1234"
assert extract_gsm8k_answer("The net change is #### -7") == "-7"
assert extract_gsm8k_answer("So the answer is #### 3.50") == "3.50"
assert extract_gsm8k_answer("...so the total is 18 apples.") == "18"
assert extract_gsm8k_answer("") is None
assert is_correct("3.0", "3") is True
assert is_correct("1234", "1,234") is True
assert is_correct(None, "5") is False
print("Parser sanity checks passed.")


In [ ]:
from datasets import load_dataset

_ds_full = load_dataset("openai/gsm8k", "main", split="test")
_rng = np.random.default_rng(SEED)
_indices = sorted(
    _rng.choice(len(_ds_full), size=min(MAX_EXAMPLES, len(_ds_full)), replace=False).tolist()
)
ds_subset = _ds_full.select(_indices)
examples = [
    {"question": row["question"],
     "ground_truth": extract_gsm8k_answer(row["answer"]),
     "answer_full": row["answer"]}
    for row in ds_subset
]
print(f"GSM8K test subset: {len(examples)} examples (seed={SEED})")


In [ ]:
def generate_once(prompt: str) -> str:
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


def _reseed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)


def run_eval_with_hooks(label, register_fn, desc_prefix=""):
    '''register_fn(model) -> list of hook handles (or []). Runs paired GSM8K subset.'''
    _reseed()
    handles = register_fn(model) if register_fn is not None else []
    results = []
    try:
        for i, ex in enumerate(tqdm(examples, desc=f"{desc_prefix}{label}")):
            prompt = f"Question: {ex['question']}\n\nLet's think step by step.\n\n"
            text = generate_once(prompt)
            pred = extract_gsm8k_answer(text)
            ok = is_correct(pred, ex["ground_truth"])
            results.append({
                "idx": i,
                "question": ex["question"],
                "ground_truth": ex["ground_truth"],
                "predicted": pred,
                "correct": ok,
                "full_output": text[-1000:],
            })
    finally:
        remove_hooks(handles)
    return results, compute_metrics(results)


def save_run(label, results, metrics, extra_state=None):
    (RESULTS_DIR / f"{label}_results.json").write_text(json.dumps(results, indent=2))
    (RESULTS_DIR / f"{label}_generations.txt").write_text(
        "\n\n====\n\n".join(
            f"[{r['idx']}] gt={r['ground_truth']} pred={r['predicted']} correct={r['correct']}\n{r['full_output']}"
            for r in results
        )
    )
    state = {
        "label": label,
        "model": MODEL_NAME,
        "seed": SEED,
        "max_examples": len(examples),
        "max_new_tokens": MAX_NEW_TOKENS,
        "device": DEVICE,
        "run_tag": RUN_TAG,
        "metrics": metrics,
    }
    if extra_state:
        state.update(extra_state)
    (RESULTS_DIR / f"{label}_state.json").write_text(json.dumps(state, indent=2))
    return state


## 9. Base eval

In [ ]:
base_cache = RESULTS_DIR / "base_results.json"
if base_cache.exists() and not FORCE_RERUN:
    base_results = json.loads(base_cache.read_text())
    base_metrics = compute_metrics(base_results)
else:
    base_results, base_metrics = run_eval_with_hooks("base", register_fn=None)
    save_run("base", base_results, base_metrics)
print(f"base: acc={base_metrics['accuracy']:.3f}")


## 10. Sweep (layer, factor)

In [ ]:
grid = {}

for L in LAYERS:
    for factor in FACTORS:
        label = f"L{L}_f{factor}"
        strength = factor - 1.0
        cache = RESULTS_DIR / f"{label}_results.json"
        if cache.exists() and not FORCE_RERUN:
            res = json.loads(cache.read_text())
            met = compute_metrics(res)
            print(f"[cached] {label}: acc={met['accuracy']:.3f}")
        else:
            vec_tensor = probes[L]["tensor"]
            def reg(mdl, vt=vec_tensor, s=strength, lyr=L):
                return register_steering(mdl, lyr, vt, s)
            res, met = run_eval_with_hooks(label, register_fn=reg)
            save_run(label, res, met, extra_state={
                "layer": L, "factor": factor, "strength": strength,
                "vector_norm": probes[L]["cfg"]["vector_norm"],
            })
            print(f"[done]   {label}: acc={met['accuracy']:.3f}")
        grid[(L, factor)] = met


## 11. Grouped-bar summary + heatmap

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110

rows = [{"layer": L, "factor": f, "accuracy": grid[(L, f)]["accuracy"],
         "f1": grid[(L, f)]["f1"], "parse_rate": grid[(L, f)]["parse_rate"]}
        for L in LAYERS for f in FACTORS]
df = pd.DataFrame(rows)
df.to_csv(RESULTS_DIR / "summaries.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
xs = np.arange(len(LAYERS))
w = 0.8 / len(FACTORS)
for i, f in enumerate(FACTORS):
    vals = [grid[(L, f)]["accuracy"] for L in LAYERS]
    ax.bar(xs + i * w - 0.4 + w/2, vals, w, label=f"factor {f}")
ax.axhline(base_metrics["accuracy"], ls="--", color="gray", label="base")
ax.set_xticks(xs); ax.set_xticklabels([f"L{L}" for L in LAYERS])
ax.set_ylabel("accuracy"); ax.set_title("GSM8K accuracy: layer x factor")
ax.legend(); fig.tight_layout()
fig.savefig(RESULTS_DIR / "layer_sweep.png"); plt.show()
df.pivot(index="layer", columns="factor", values="accuracy")


## 13. Intervene-to-probe range (Issue #6)


In [ ]:
# === Intervene-to-probe-range arm (Issue #6 in docs/issues.md) ===
# Instead of intervening at one layer, register the SAME vector at every layer
# in [intervene_layer, probe_layer]. This is the GoT recommendation: an
# intervention at layer k that's still readable by the layer-L probe must
# *propagate* causally through layers k..L; the range-arm makes that
# propagation cheap by re-applying the perturbation at every step.
import torch

PROBE_LAYER = max(LAYERS)  # downstream-most loaded probe
RANGE_OFFSETS = (2, 4, 6)
range_metrics = {}

for off in RANGE_OFFSETS:
    intervene_layer = PROBE_LAYER - off
    if intervene_layer < 1:
        continue
    spec = []
    for L in range(intervene_layer, PROBE_LAYER + 1):
        if L in probes:
            spec.append((L, probes[L]["tensor"], 0.2))
    if len(spec) < 2:
        continue
    label = f"range_off{off}_probe{PROBE_LAYER}"
    handles = register_additive_hooks(model, spec, mode="additive_normalized")
    try:
        results = []
        for i, ex in enumerate(examples):
            prompt = f"Question: {ex['question']}\n\nLet's think step by step.\n\n"
            text = generate_once(prompt)
            pred = extract_gsm8k_answer(text)
            ok = is_correct(pred, ex["ground_truth"])
            results.append({"idx": i, "predicted": pred, "correct": ok,
                            "ground_truth": ex["ground_truth"], "full_output": text[-400:]})
        met = compute_metrics(results)
        range_metrics[label] = met
        save_run(label, results, met, extra_state={
            "arm": "range",
            "intervene_layers": [L for L, _, _ in spec],
            "probe_layer": PROBE_LAYER,
            "offset": off,
            "hook_target_layer_idx": [L - 1 for L, _, _ in spec],
            "mode": "additive_normalized",
            "vector_source": "caa_train_per_layer",
            "convention_check_passed": bool(globals().get("CONVENTION_CHECK_PASSED", False)),
        })
        print(f"  {label} (layers {[L for L,_,_ in spec]}): acc={met['accuracy']:.3f}")
    finally:
        remove_hooks(handles)

(RESULTS_DIR / "range_metrics.json").write_text(json.dumps(range_metrics, indent=2))


## 12. Bundle

In [ ]:
zip_path = Path(f"nb_results_{RUN_TAG}.zip")
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in RESULTS_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(RESULTS_DIR.parent))
print(f"Zipped {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)")

try:
    from google.colab import files  # type: ignore
    files.download(str(zip_path))
except Exception:
    print("Not on Colab — the zip is on the local filesystem.")
